[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab12.ipynb)

# Bab 12 — AI yang Dapat Dijelaskan, *Shortcut Learning*, dan Ketidakpastian
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Jalankan di **Google Colab dengan GPU T4**, lalu **Runtime → Run all**.

Isi (PneumoniaMNIST 128 px, CC BY 4.0):
1. **Grad-CAM** (ditulis sendiri dengan *hook*), *saliency* gradien, dan *occlusion*; uji kewarasan acak-bobot (Adebayo dkk., 2018).
2. **Demo *shortcut learning***: penanda persegi **sintetis** ditempel pada 95 % citra pneumonia dan 5 % citra normal di set latih/validasi.
   Model tampak hebat pada data "serumah", lalu runtuh saat penanda dihapus atau dibalik — dan Grad-CAM menyorot penanda.
3. **Kinerja per subkelompok**: PneumoniaMNIST tidak memuat data demografi; sebagai peragaan dipakai subkelompok teknis (tertil kecerahan citra).
4. **Ketidakpastian**: MC dropout (T = 30) dan *deep ensemble* (4 model) — AUC, ECE, NLL, kurva *selective prediction*,
   dan entropi pada citra di luar distribusi (irisan CT OrganAMNIST).

Hasil → `/content/hasil_bab12/` → `/content/hasil_bab12.zip`. Hanya untuk pembelajaran, **bukan** untuk penggunaan klinis.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 12.0 Lingkungan

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil, random
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # perlu untuk algoritme deterministik di GPU
T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB12_SMOKE") == "1"          # uji cepat penulis di luar Colab (CPU, subset kecil); di Colab = False
OUT = Path(os.environ.get("BAB12_OUT", "/content/hasil_bab12"))
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    WAKTU[nama] = round(time.perf_counter() - t0, 2)
    print(f"[waktu] {nama}: {WAKTU[nama]:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?). Di Colab: Runtime > Change runtime type > T4 GPU.")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'"); info_mem = jalankan("free -h | head -2"); info_disk = jalankan("df -h /content 2>/dev/null || df -h /")
    print(info_cpu, info_mem, info_disk)
    (OUT / "info_lingkungan.txt").write_text(f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}\n{info_disk}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

In [ ]:
with catat_waktu("1_pip_install"):
    jalankan('pip install -q "medmnist==3.0.2"')
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib, matplotlib.ticker; import matplotlib.pyplot as plt
import torchvision, sklearn, medmnist
from medmnist import INFO
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
         "numpy": np.__version__, "scikit-learn": sklearn.__version__, "medmnist": medmnist.__version__,
         "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(json.dumps(VERSI, indent=1))
def npz_medmnist(flag, size):
    fn = berkas_medmnist(flag, size); folder = pastikan_dataset("medmnist", [fn]); return np.load(folder / fn)
SEED = 42
def seed_semua(seed=SEED):
    """Seed Python, NumPy, PyTorch (CPU+GPU) dan minta algoritme deterministik."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_semua()
# ---------- Perangkat latih bersama (sama di notebook bab 04 dst.) ----------
MEAN_IN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); SD_IN = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
AMP = DEV.type == "cuda"

def ke_tensor_uint8(X):
    """(N,H,W) atau (N,H,W,C) uint8 -> tensor (N,C,H,W) uint8; disimpan di GPU bila ada (data kecil)."""
    t = torch.as_tensor(X)
    t = t[:, None] if t.ndim == 3 else t.permute(0, 3, 1, 2)
    return t.contiguous().to(DEV)

def augmentasi_gpu(x, flip_h=False, flip_v=False, rot90=False, rot_derajat=0.0, geser=0.0, kecerahan=0.1, kontras=0.1, warna=0.0):
    """x float (N,C,H,W) di [0,1]. Semua acak per sampel kecuali rot90 (per batch)."""
    n = x.shape[0]; d = x.device
    if flip_h: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(3), x)
    if flip_v: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(2), x)
    if rot90: x = torch.rot90(x, int(torch.randint(0, 4, (1,)).item()), (2, 3))
    if rot_derajat or geser:
        a = (torch.rand(n, device=d) * 2 - 1) * rot_derajat * np.pi / 180
        t = (torch.rand(n, 2, device=d) * 2 - 1) * geser * 2
        th = torch.stack([torch.stack([a.cos(), -a.sin(), t[:, 0]], 1), torch.stack([a.sin(), a.cos(), t[:, 1]], 1)], 1)
        x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode="border", align_corners=False)
    if warna and x.shape[1] == 3: x = x * (1 + (torch.rand(n, 3, 1, 1, device=d) * 2 - 1) * warna)
    m = x.mean((1, 2, 3), keepdim=True)
    x = ((x - m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kontras) + m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kecerahan)
    return x.clamp(0, 1)

class ResNet18(nn.Module):
    """ResNet-18 torchvision; masukan 1 atau 3 kanal [0,1], opsional diperbesar ke `uk` piksel, normalisasi ImageNet."""
    def __init__(self, k, pretrain=True, uk=None, p_drop=0.0):
        super().__init__(); self.uk = uk
        w = None
        if pretrain:
            w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
            try: self.net = torchvision.models.resnet18(weights=w)
            except Exception as e:
                if not SMOKE: raise
                print("SMOKE: bobot ImageNet gagal diunduh, pakai bobot acak:", e); self.net = torchvision.models.resnet18(weights=None)
        else: self.net = torchvision.models.resnet18(weights=None)
        self.net.fc = nn.Sequential(nn.Dropout(p_drop), nn.Linear(self.net.fc.in_features, k))
    def forward(self, x):
        if x.shape[1] == 1: x = x.repeat(1, 3, 1, 1)
        if self.uk and x.shape[-1] != self.uk: x = F.interpolate(x, size=(self.uk, self.uk), mode="bilinear", align_corners=False)
        return self.net((x - MEAN_IN.to(x.device)) / SD_IN.to(x.device))

@torch.no_grad()
def logit_semua(model, X, bs=512, praproses=None):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].to(DEV).float().div(255)
        if praproses: x = praproses(x)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): out.append(model(x).float())
    return torch.cat(out)

def latih(nama, model, X, Y, Xv, Yv, fungsi_loss, skor_val, maks_epoch=30, patience=5, aug=None, lr=1e-3, bs=128,
          tahap=None, bobot_sampel=None, praproses=None, seed=SEED, wd=1e-4):
    """Latih dengan AMP; early stopping memantau skor_val(logit_val, y_val) (lebih tinggi lebih baik).
    tahap: daftar (n_epoch, lr, beku_backbone). bobot_sampel: oversampling (sampling dengan pengembalian)."""
    seed_semua(seed); model = model.to(DEV); Yd = torch.as_tensor(Y).to(DEV); Yvd = torch.as_tensor(Yv).to(DEV)
    if SMOKE: maks_epoch, patience = min(maks_epoch, 2), 1
    scaler = torch.amp.GradScaler("cuda", enabled=AMP); log = []; best = (-np.inf, None, 0); ep_g = 0
    g = torch.Generator(device="cpu").manual_seed(seed)
    for n_ep, lr_t, beku in (tahap or [(maks_epoch, lr, False)]):
        if SMOKE: n_ep = min(n_ep, 2)
        for nm, p in model.named_parameters(): p.requires_grad = (not beku) or nm.startswith("net.fc")
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_t, weight_decay=wd); tunggu = 0
        for ep in range(n_ep):
            model.train(); t0 = time.perf_counter()
            if bobot_sampel is not None: perm = torch.multinomial(torch.as_tensor(bobot_sampel, dtype=torch.float), len(Y), replacement=True, generator=g)
            else: perm = torch.randperm(len(Y), generator=g)
            perm = perm.to(X.device); tl = 0.0; n = 0
            for i in range(0, len(perm), bs):
                b = perm[i:i + bs]; x = X[b].to(DEV).float().div(255)
                if aug: x = augmentasi_gpu(x, **aug)
                if praproses: x = praproses(x)
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    lg = model(x)
                loss = fungsi_loss(lg.float(), Yd[b.to(DEV)])
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tl += loss.item() * len(b); n += len(b)
            if DEV.type == "cuda": torch.cuda.synchronize()
            dt = time.perf_counter() - t0; ep_g += 1
            lv = logit_semua(model, Xv, praproses=praproses); vl = fungsi_loss(lv, Yvd).item(); sv = float(skor_val(lv, Yv))
            log.append({"model": nama, "epoch": ep_g, "tahap_beku": beku, "loss_latih": tl / n, "loss_val": vl, "skor_val": sv, "detik_epoch": dt})
            print(f"{nama} ep{ep_g:02d} loss {tl/n:.4f}/{vl:.4f} skor_val {sv:.4f} {dt:.1f}s")
            if sv > best[0]: best = (sv, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep_g); tunggu = 0
            else:
                tunggu += 1
                if not beku and tunggu >= patience: print(f"  early stopping (patience {patience})"); break
    model.load_state_dict(best[1])
    return model, pd.DataFrame(log), best[2]

def n_param(m): return sum(p.numel() for p in m.parameters())

def ci_bootstrap(fn, y, p, n_boot=1000, seed=SEED, strata=True):
    """CI 95 % persentil; resampling bertingkat per kelas agar setiap sampel memuat semua kelas."""
    r = np.random.default_rng(seed); y = np.asarray(y); idx_k = [np.where(y == k)[0] for k in np.unique(y)]; v = []
    for _ in range(n_boot):
        i = np.concatenate([r.choice(ik, len(ik)) for ik in idx_k]) if strata else r.choice(len(y), len(y))
        try: v.append(fn(y[i], p[i]))
        except ValueError: pass
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def gambar_kurva(LOG, judul_skor, path):
    k = len(LOG); fig, ax = plt.subplots(1, k, figsize=(4.2 * k, 3.2), squeeze=False); ax = ax[0]
    for a, lg in zip(ax, LOG):
        a.plot(lg.epoch, lg.loss_latih, "o-", ms=3, label="loss latih"); a.plot(lg.epoch, lg.loss_val, "s-", ms=3, label="loss val")
        a2 = a.twinx(); a2.plot(lg.epoch, lg.skor_val, "^-", ms=3, color="tab:green", label=judul_skor); a2.set_ylabel(judul_skor, color="tab:green")
        eb = int(lg.loc[lg.skor_val.idxmax(), "epoch"]); a.axvline(eb, color="red", ls="--", lw=1, label=f"terbaik ep {eb}")
        if lg.tahap_beku.any(): a.axvspan(0.5, lg[lg.tahap_beku].epoch.max() + 0.5, color="gray", alpha=0.15, label="tahap beku")
        a.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True)); a.set_title(lg.model.iloc[0], fontsize=8); a.set_xlabel("epoch"); a.legend(fontsize=6)
    plt.tight_layout(); plt.savefig(path, bbox_inches="tight"); plt.show()
from sklearn.metrics import roc_auc_score

## 12.1 Data dan model bersih (ResNet-18, dropout 0,3 di kepala)

In [ ]:
UK = 28 if SMOKE else 128
with catat_waktu("2_muat_data"):
    d = npz_medmnist("pneumoniamnist", UK)
    D = {s: (d[f"{s}_images"], d[f"{s}_labels"].reshape(-1).astype(np.int64)) for s in ("train", "val", "test")}
    if SMOKE:
        r = np.random.default_rng(0); D = {s: (x[i], y[i]) for s, (x, y) in D.items() for i in [r.choice(len(y), min(len(y), 300), replace=False)]}
XT = {s: ke_tensor_uint8(D[s][0]) for s in D}; Y = {s: D[s][1] for s in D}; yt = Y["test"]
loss_bce = lambda lg, y: F.binary_cross_entropy_with_logits(lg[:, 0], y.float())
def skor_auc(lg, y): return roc_auc_score(y, lg[:, 0].cpu().numpy())
AUG = dict(rot_derajat=7, geser=0.05, kecerahan=0.1, kontras=0.1)
TAHAP = [(2, 1e-3, True), (10, 1e-4, False)]
def latih_cepat(nama, X, Yy, Xv, Yv, seed): return latih(nama, ResNet18(1, True, p_drop=0.3), X, Yy, Xv, Yv, loss_bce, skor_auc, patience=4, aug=AUG, tahap=TAHAP, bs=64, seed=seed)[0]
with catat_waktu("latih_bersih_s0"): M_BERSIH = latih_cepat("bersih | seed 0", XT["train"], Y["train"], XT["val"], Y["val"], 0)
sig = lambda z: 1 / (1 + np.exp(-z))
P_BERSIH = sig(logit_semua(M_BERSIH, XT["test"])[:, 0].cpu().numpy().astype(np.float64)); print("AUC uji model bersih:", roc_auc_score(yt, P_BERSIH))

## 12.2 Grad-CAM, *saliency*, dan *occlusion*
Grad-CAM: bobot kanal = rerata gradien logit terhadap peta aktivasi `layer3` (8×8 pada masukan 128 px; `layer4` hanya 4×4 sehingga terlalu kasar); peta = ReLU(Σ bobot × aktivasi), diperbesar ke ukuran citra.
Untuk prediksi normal, gradien diambil dari **−logit** (bukti ke arah kelas normal).

In [ ]:
def gradcam(model, x, layer):
    simpan_ = {}
    def fwd(m, i, o): simpan_["a"] = o; o.register_hook(lambda g: simpan_.__setitem__("g", g))
    h = layer.register_forward_hook(fwd); model.eval()
    x = x.float().div(255).requires_grad_(True); lg = model(x)[:, 0]
    tanda = torch.where(lg >= 0, 1.0, -1.0); model.zero_grad(); (lg * tanda).sum().backward(); h.remove()
    w = simpan_["g"].mean((2, 3), keepdim=True); cam = F.relu((w * simpan_["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam = cam / (cam.amax((1, 2), keepdim=True) + 1e-8); return cam.detach().cpu().numpy(), torch.sigmoid(lg).detach().cpu().numpy()
def saliency(model, x):
    model.eval(); x = x.float().div(255).requires_grad_(True); lg = model(x)[:, 0]; tanda = torch.where(lg >= 0, 1.0, -1.0)
    (lg * tanda).sum().backward(); s = x.grad.abs()[:, 0]; return (s / (s.amax((1, 2), keepdim=True) + 1e-8)).cpu().numpy()
@torch.no_grad()
def occlusion(model, x, patch=None, stride=None):
    model.eval(); n, _, H, W = x.shape; patch = patch or H // 8; stride = stride or patch // 2; x = x.float().div(255)
    base = torch.sigmoid(model(x)[:, 0]); peta = torch.zeros(n, H, W, device=x.device); hit = torch.zeros(H, W, device=x.device)
    for i in range(0, H - patch + 1, stride):
        for j in range(0, W - patch + 1, stride):
            xo = x.clone(); xo[:, :, i:i + patch, j:j + patch] = x.mean((2, 3), keepdim=True)
            dp = base - torch.sigmoid(model(xo)[:, 0]); peta[:, i:i + patch, j:j + patch] += dp.abs()[:, None, None]; hit[i:i + patch, j:j + patch] += 1
    peta = peta / hit.clamp(min=1); return (peta / (peta.amax((1, 2), keepdim=True) + 1e-8)).cpu().numpy()
yh = P_BERSIH >= 0.5
pilih = {"TP": np.where(yh & (yt == 1))[0][0], "TN": np.where(~yh & (yt == 0))[0][0],
         "FP": np.where(yh & (yt == 0))[0][0] if (yh & (yt == 0)).any() else 0, "FN": np.where(~yh & (yt == 1))[0][0] if (~yh & (yt == 1)).any() else 1}
idx = list(pilih.values()); xs = XT["test"][idx]
with catat_waktu("3_xai"):
    cam, pp = gradcam(M_BERSIH, xs, M_BERSIH.net.layer3); sal = saliency(M_BERSIH, xs); occ = occlusion(M_BERSIH, xs)
    m_acak = ResNet18(1, False, p_drop=0.3).to(DEV); cam_acak, _ = gradcam(m_acak, xs, m_acak.net.layer3)    # uji kewarasan: bobot acak
fig, ax = plt.subplots(4, 5, figsize=(12, 10))
for r_, (nm, i) in enumerate(pilih.items()):
    img = XT["test"][i, 0].cpu().numpy()
    for c_, (judul, peta) in enumerate([("citra", None), ("Grad-CAM", cam[r_]), ("saliency", sal[r_]), ("occlusion", occ[r_]), ("Grad-CAM bobot ACAK", cam_acak[r_])]):
        a = ax[r_, c_]; a.imshow(img, cmap="gray"); a.axis("off")
        if peta is not None: a.imshow(peta, cmap="jet", alpha=0.45)
        a.set_title(f"{nm}: p={pp[r_]:.2f}" if c_ == 0 else judul, fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "gambar_12_1_xai_bersih.png", bbox_inches="tight"); plt.show()

## 12.3 Demo *shortcut learning* dengan penanda sintetis
Penanda = persegi putih bertepi hitam di pojok kiri atas (mirip penanda lateralitas/teks alat). Ini **sintetis**, ditambahkan oleh kode ini.
- Set latih & validasi "bias": penanda pada 95 % pneumonia dan 5 % normal.
- Set uji: (a) **bersih** tanpa penanda, (b) **bias sama** seperti latih, (c) **dibalik** (penanda pada 95 % normal dan 5 % pneumonia).

In [ ]:
S = max(4, UK // 10); O = max(2, UK // 32)
def tempel(X, mask):
    X = X.clone(); i = torch.as_tensor(np.where(mask)[0], device=X.device)
    X[i, :, O:O + S, O:O + S] = 255; X[i, :, O + S // 4:O + S - S // 4, O + S // 4:O + S - S // 4] = 0; return X
def mask_bias(y, p_pos, p_neg, seed): r = np.random.default_rng(seed); u = r.random(len(y)); return np.where(y == 1, u < p_pos, u < p_neg)
XB = {"train": tempel(XT["train"], mask_bias(Y["train"], 0.95, 0.05, 1)), "val": tempel(XT["val"], mask_bias(Y["val"], 0.95, 0.05, 2))}
UJI = {"bersih": XT["test"], "bias sama": tempel(XT["test"], mask_bias(yt, 0.95, 0.05, 3)), "dibalik": tempel(XT["test"], mask_bias(yt, 0.05, 0.95, 3))}
with catat_waktu("latih_bias"): M_BIAS = latih_cepat("bias penanda | seed 0", XB["train"], Y["train"], XB["val"], Y["val"], 0)
baris = []
for nm_m, m in [("model bersih", M_BERSIH), ("model latih-bias", M_BIAS)]:
    for nm_u, X in UJI.items():
        p = sig(logit_semua(m, X)[:, 0].cpu().numpy().astype(np.float64)); yh_ = p >= 0.5
        baris.append({"model": nm_m, "set_uji": nm_u, "AUC": roc_auc_score(yt, p), "akurasi@0.5": float((yh_ == yt).mean()),
                      "sensitivitas@0.5": float(yh_[yt == 1].mean()), "spesifisitas@0.5": float((~yh_[yt == 0]).mean())})
T121 = pd.DataFrame(baris); T121.to_csv(OUT / "tabel_12_1_shortcut.csv", index=False)
# Fraksi "energi" Grad-CAM di dalam area penanda (citra uji bias sama yang berpenanda)
ber = np.where(mask_bias(yt, 0.95, 0.05, 3))[0][:64]; kotak = np.zeros((UK, UK), bool); kotak[O:O + S, O:O + S] = True
EN = {}
for nm_m, m in [("model bersih", M_BERSIH), ("model latih-bias", M_BIAS)]:
    c_, _ = gradcam(m, UJI["bias sama"][ber], m.net.layer3); EN[nm_m] = float((c_[:, kotak].sum(1) / (c_.reshape(len(ber), -1).sum(1) + 1e-8)).mean())
EN["luas_penanda_relatif"] = float(kotak.mean()); print("fraksi energi Grad-CAM di area penanda:", EN)
fig, ax = plt.subplots(2, 4, figsize=(11, 5.6)); contoh = ber[:4]
for c_, i in enumerate(contoh):
    for r_, (nm_m, m) in enumerate([("bersih", M_BERSIH), ("latih-bias", M_BIAS)]):
        cm, pp = gradcam(m, UJI["bias sama"][i:i + 1], m.net.layer3)
        ax[r_, c_].imshow(UJI["bias sama"][i, 0].cpu(), cmap="gray"); ax[r_, c_].imshow(cm[0], cmap="jet", alpha=0.45); ax[r_, c_].axis("off")
        ax[r_, c_].set_title(f"model {nm_m}: p={pp[0]:.2f} (label {yt[i]})", fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "gambar_12_2_shortcut_gradcam.png", bbox_inches="tight"); plt.show()
fig, a = plt.subplots(figsize=(6, 3.2)); w = 0.35
for k, nm_m in enumerate(["model bersih", "model latih-bias"]):
    d_ = T121[T121.model == nm_m]; a.bar(np.arange(3) + k * w, d_.AUC, w, label=nm_m)
a.set_xticks(np.arange(3) + w / 2); a.set_xticklabels(list(UJI)); a.set_ylabel("AUC uji"); a.set_ylim(0, 1); a.axhline(0.5, color="gray", ls=":"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "gambar_12_3_shortcut_auc.png", bbox_inches="tight"); plt.show(); T121

## 12.4 Kinerja per subkelompok (peragaan)
Tanpa atribut demografis, kita **tidak dapat** menilai keadilan terhadap jenis kelamin, usia, atau etnis pada dataset ini.
Sebagai peragaan cara kerjanya, set uji dibagi menjadi tertil **kecerahan rerata citra** (proksi kasar kondisi akuisisi).

In [ ]:
kec = XT["test"].float().mean((1, 2, 3)).cpu().numpy(); ter = pd.qcut(kec, 3, labels=["gelap", "sedang", "terang"])
baris = []
for g in ["gelap", "sedang", "terang"]:
    i = np.where(ter == g)[0]; y_, p_ = yt[i], P_BERSIH[i]; yh_ = p_ >= 0.5
    baris.append({"subkelompok": g, "n": len(i), "prevalensi": float(y_.mean()), "AUC": roc_auc_score(y_, p_) if len(np.unique(y_)) > 1 else np.nan,
                  "AUC_CI95": ci_bootstrap(roc_auc_score, y_, p_, n_boot=200 if SMOKE else 1000) if len(np.unique(y_)) > 1 else None,
                  "sensitivitas@0.5": float(yh_[y_ == 1].mean()) if (y_ == 1).any() else np.nan, "spesifisitas@0.5": float((~yh_[y_ == 0]).mean()) if (y_ == 0).any() else np.nan})
T122 = pd.DataFrame(baris); T122.to_csv(OUT / "tabel_12_2_subkelompok.csv", index=False); T122

## 12.5 Ketidakpastian: MC dropout dan *deep ensemble*
- **Model tunggal**: dropout nonaktif saat inferensi.
- **MC dropout**: dropout di kepala tetap aktif, 30 kali *forward*, probabilitas dirata-rata.
- **Deep ensemble**: 4 model identik dengan seed berbeda (0–3), probabilitas dirata-rata.
Ketidakpastian = entropi prediktif biner. *Selective prediction*: kasus paling tidak pasti dirujuk ke klinisi, akurasi dihitung pada sisanya.

In [ ]:
@torch.no_grad()
def mc_dropout(model, X, T=30):
    model.eval()
    for mod in model.modules():
        if isinstance(mod, nn.Dropout): mod.train()
    p = np.mean([sig(logit_semua_mc(model, X)) for _ in range(T)], 0); model.eval(); return p
@torch.no_grad()
def logit_semua_mc(model, X, bs=512):          # seperti logit_semua tetapi TIDAK memanggil model.eval()
    out = []
    for i in range(0, len(X), bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): out.append(model(X[i:i + bs].float().div(255)).float())
    return torch.cat(out)[:, 0].cpu().numpy().astype(np.float64)
ANGGOTA = [M_BERSIH]
for s in ([1] if SMOKE else [1, 2, 3]):
    with catat_waktu(f"latih_ensemble_s{s}"): ANGGOTA.append(latih_cepat(f"ensemble | seed {s}", XT["train"], Y["train"], XT["val"], Y["val"], s))
def ece(y, p, nb=15):
    b = np.minimum((p * nb).astype(int), nb - 1); return float(sum(abs(y[b == k].mean() - p[b == k].mean()) * (b == k).mean() for k in range(nb) if (b == k).any()))
def nll(y, p): p = np.clip(p, 1e-7, 1 - 1e-7); return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
def entropi(p): p = np.clip(p, 1e-7, 1 - 1e-7); return -(p * np.log(p) + (1 - p) * np.log(1 - p))
with catat_waktu("4_ketidakpastian"):
    PRED = {"model tunggal": P_BERSIH, "MC dropout (T=30)": mc_dropout(M_BERSIH, XT["test"]),
            f"deep ensemble ({len(ANGGOTA)} model)": np.mean([sig(logit_semua(m, XT["test"])[:, 0].cpu().numpy().astype(np.float64)) for m in ANGGOTA], 0)}
    # OOD: irisan CT OrganAMNIST (CC BY 4.0) diperbesar ke UK
    do = npz_medmnist("organamnist", 28); xo = torch.as_tensor(do["test_images"][:500])[:, None].float()
    XOOD = F.interpolate(xo, size=(UK, UK), mode="bilinear", align_corners=False).round().clamp(0, 255).to(torch.uint8).to(DEV)
    POOD = {"model tunggal": sig(logit_semua(M_BERSIH, XOOD)[:, 0].cpu().numpy().astype(np.float64)), "MC dropout (T=30)": mc_dropout(M_BERSIH, XOOD),
            f"deep ensemble ({len(ANGGOTA)} model)": np.mean([sig(logit_semua(m, XOOD)[:, 0].cpu().numpy().astype(np.float64)) for m in ANGGOTA], 0)}
baris = []; fig, ax = plt.subplots(1, 2, figsize=(11, 3.8)); cakupan = np.linspace(0.5, 1, 11)
for nm, p in PRED.items():
    u = entropi(p); salah = ((p >= 0.5) != yt); urut = np.argsort(u)
    akur = [1 - salah[urut[:max(1, int(round(c * len(p))))]].mean() for c in cakupan]
    uo = entropi(POOD[nm]); auc_ood = roc_auc_score(np.r_[np.zeros(len(u)), np.ones(len(uo))], np.r_[u, uo])
    baris.append({"metode": nm, "AUC": roc_auc_score(yt, p), "akurasi@0.5": float(1 - salah.mean()), "ECE": ece(yt, p), "NLL": nll(yt, p),
                  "AUROC_ketidakpastian_vs_salah": roc_auc_score(salah, u) if 0 < salah.sum() < len(salah) else np.nan,
                  "akurasi_cakupan_80%": float(akur[list(np.round(cakupan, 2)).index(0.8)]), "entropi_rerata_ID": float(u.mean()), "entropi_rerata_OOD": float(uo.mean()),
                  "AUROC_deteksi_OOD_entropi": float(auc_ood), "persen_OOD_diprediksi_pneumonia": float((POOD[nm] >= 0.5).mean() * 100)})
    ax[0].plot(cakupan, akur, "o-", ms=3, label=nm); ax[1].hist(uo, bins=30, alpha=0.4, label=f"OOD – {nm}", density=True)
ax[1].hist(entropi(PRED["model tunggal"]), bins=30, alpha=0.4, label="ID (uji) – model tunggal", density=True, histtype="step", color="k")
ax[0].set_xlabel("cakupan (fraksi kasus yang diputuskan model)"); ax[0].set_ylabel("akurasi pada kasus yang diputuskan"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("entropi prediktif"); ax[1].set_ylabel("kepadatan"); ax[1].legend(fontsize=6)
plt.tight_layout(); plt.savefig(OUT / "gambar_12_4_ketidakpastian.png", bbox_inches="tight"); plt.show()
T123 = pd.DataFrame(baris); T123.to_csv(OUT / "tabel_12_3_ketidakpastian.csv", index=False); T123

## 12.6 Ringkasan, zip hasil, dan total waktu

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {str(k): bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 6)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada GPU (CPU)"
RINGKASAN = bersih(dict(smoke=SMOKE, gpu=GPU, versi=VERSI, waktu_detik=WAKTU, **(dict(ukuran_px=UK, shortcut=T121.to_dict("records"), energi_gradcam_penanda=EN, subkelompok=T122.to_dict("records"), ketidakpastian=T123.to_dict("records")))))
(OUT / "ringkasan_bab12.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
print(f"TOTAL WAKTU NOTEBOOK: {WAKTU['total_notebook']:.1f} s ({WAKTU['total_notebook']/60:.1f} menit) | GPU: {GPU}")
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass